# Практическая работа №2: Визуализация многомерных данных (Лица Чернова)
**СПбГЭТУ «ЛЭТИ», 2026 г.**

**Автор:** Шумов О.Д.

**Вариант:** 1 (номер студенческого 230025)

**Зона:** F1_Z2_***

**Описание:**
Данный скрипт загружает данные HVAC из файла bldg-MC2.csv,
выполняет анализ (статистики, корреляции), нормализацию,
и строит 14 лиц Чернова (по одному на каждый день) для визуализации
многомерных параметров микроклимата в заданной зоне (F1_Z2).

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle, Polygon, Arc
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
import os

print("="*60)
print("Практическая работа №2. Вариант 1 (F1_Z2)")
print("="*60)

## 1. Загрузка и предварительная обработка данных

In [ ]:
# Укажите путь к вашему файлу данных
FILE_PATH = 'bldg-MC2.csv'  # <-- УБЕДИТЕСЬ, ЧТО ФАЙЛ В ПАПКЕ С БЛОКНОТОМ!

# Загружаем данные
print("\n1. Загрузка данных...")
try:
    # Пытаемся прочитать с правильным разделителем (в файле запятые)
    df = pd.read_csv(FILE_PATH, encoding='utf-8')
    print(f"   Файл загружен. Размерность: {df.shape}")
except FileNotFoundError:
    print(f"   ОШИБКА: Файл {FILE_PATH} не найден в текущей папке.")
    print("   Пожалуйста, поместите файл в ту же папку, что и этот блокнот, или укажите полный путь.")
    # Выходим, если файл не найден
    raise

# Определяем префикс нашей зоны по варианту 1
# ВАЖНО: В файле названия начинаются с пробела!
zone_prefix = ' F_1_Z_2'  # Добавляем пробел в начале!
print(f"\n2. Анализируемая зона: {zone_prefix}")

# Отбираем колонки, относящиеся к нашей зоне
all_columns = df.columns.tolist()
print(f"   Первые несколько колонок: {all_columns[:5]}")

# Ищем все колонки с нашим префиксом
zone_columns = [col for col in all_columns if col.startswith(zone_prefix)]

# Если не нашли, пробуем без пробела
if not zone_columns:
    print("   Не найдено с пробелом, пробуем без пробела...")
    zone_prefix_alt = 'F_1_Z_2'
    zone_columns = [col for col in all_columns if col.startswith(zone_prefix_alt)]
    
    if zone_columns:
        print(f"   Найдены колонки с префиксом: {zone_prefix_alt}")
        zone_prefix = zone_prefix_alt
    else:
        # Если все еще не нашли, ищем любые колонки с F_1_Z_2 в названии
        print("   Ищем частичное совпадение...")
        zone_columns = [col for col in all_columns if 'F_1_Z_2' in col]
        
        if zone_columns:
            print(f"   Найдены колонки с 'F_1_Z_2' в названии")
            print(f"   Примеры: {zone_columns[:3]}")
        else:
            # Если ничего не нашли, показываем все уникальные префиксы
            print("   ОШИБКА: Колонки для зоны F1_Z2 не найдены!")
            # Находим все уникальные префиксы зон
            unique_prefixes = set()
            for col in all_columns:
                if '_Z_' in col:
                    parts = col.split('_Z_')
                    if len(parts) > 0:
                        prefix = parts[0] + '_Z_'
                        unique_prefixes.add(prefix)
            print(f"   Доступные префиксы зон: {sorted(unique_prefixes)}")
            raise ValueError(f"Не удалось найти колонки для зоны F1_Z2")

print(f"   Найдено параметров в зоне: {len(zone_columns)}")
print(f"   Параметры: {zone_columns}")

# Парсим временную метку из первого столбца (он называется 'Date/Time')
# Преобразуем в datetime
df['timestamp'] = pd.to_datetime(df['Date/Time'])
df['date'] = df['timestamp'].dt.date

## 2. Анализ данных (Статистики и корреляция)

In [ ]:
print("\n" + "="*60)
print("2. АНАЛИЗ ДАННЫХ")
print("="*60)

# Статистики по каждому параметру за весь период
stats_df = df[zone_columns].describe().T
stats_df['var'] = df[zone_columns].var()
print("\n   Основные статистики (среднее, мин, макс, дисперсия):")
print(stats_df[['mean', 'min', 'max', 'var']])

# Корреляционная матрица для понимания связей между параметрами
corr_matrix = df[zone_columns].corr()
print("\n   Корреляционная матрица:")
print(corr_matrix)

# Визуализация корреляционной матрицы (для отчета)
plt.figure(figsize=(12, 10))
plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(label='Корреляция')
# Упрощаем названия для читаемости
short_names = [col.split(':')[-1].strip() for col in zone_columns]
plt.xticks(range(len(zone_columns)), short_names, rotation=90, fontsize=8)
plt.yticks(range(len(zone_columns)), short_names, fontsize=8)
plt.title(f'Корреляционная матрица параметров зоны F1_Z2')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("   График корреляционной матрицы сохранен как 'correlation_matrix.png'")

## 3. Агрегация по дням и нормализация

In [ ]:
print("\n" + "="*60)
print("3. АГРЕГАЦИЯ ПО ДНЯМ И НОРМАЛИЗАЦИЯ")
print("="*60)

# Группируем по дням и берем среднее
daily_avg = df.groupby('date')[zone_columns].mean().reset_index()

# Сортируем по дате
daily_avg = daily_avg.sort_values('date')

print(f"\n   Агрегированные данные по дням. Форма: {daily_avg.shape}")
print(f"   Дни: {daily_avg['date'].tolist()}")

# Проверяем, что у нас есть данные
if len(daily_avg) == 0:
    raise ValueError("Нет данных после агрегации по дням!")

# Берем первые 14 дней, если их больше
if len(daily_avg) > 14:
    daily_avg = daily_avg.head(14)
    print(f"   Используем первые 14 дней (с {daily_avg['date'].min()} по {daily_avg['date'].max()})")
elif len(daily_avg) < 14:
    print(f"   ВНИМАНИЕ: Доступно только {len(daily_avg)} дней, ожидалось 14.")

dates = daily_avg['date'].tolist()

# Параметры для визуализации (исключаем колонку 'date')
feature_cols = [col for col in daily_avg.columns if col != 'date']

# Нормализация данных в диапазон [0, 1] для использования в чертах лица
scaler = MinMaxScaler(feature_range=(0, 1))
normalized_features = scaler.fit_transform(daily_avg[feature_cols])

# Создаем DataFrame с нормализованными значениями
normalized_df = pd.DataFrame(normalized_features, columns=feature_cols)
normalized_df['date'] = dates

print("\n   Нормализованные данные (первые 5 строк):")
print(normalized_df.head())

## 4. Реализация функций для рисования лица Чернова

In [ ]:
print("\n" + "="*60)
print("4. СОЗДАНИЕ ФУНКЦИЙ ДЛЯ РИСОВАНИЯ ЛИЦ")
print("="*60)

def draw_chernoff_face(ax, features_dict, title=None, day_label=None):
    """
    Рисует асимметричное лицо Чернова на указанных осях.
    
    Параметры:
    - ax: объект осей matplotlib
    - features_dict: словарь с нормализованными значениями [0,1] для каждой черты
    - title: заголовок для лица
    - day_label: метка с номером дня
    """
    # Устанавливаем пределы осей
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.axis('off')  # Отключаем оси
    
    # Извлекаем значения признаков с значениями по умолчанию
    # Размеры и формы будут изменяться от 0 до 1
    eye_size_L = features_dict.get('eye_size_L', 0.5) * 0.5 + 0.2  # [0.2, 0.7]
    eye_size_R = features_dict.get('eye_size_R', 0.5) * 0.5 + 0.2
    
    brow_slant_L = features_dict.get('brow_slant_L', 0.5) * 0.6 - 0.3  # [-0.3, 0.3]
    brow_slant_R = features_dict.get('brow_slant_R', 0.5) * 0.6 - 0.3
    
    nose_length = features_dict.get('nose_length', 0.5) * 0.5 + 0.1  # [0.1, 0.6]
    
    mouth_curve = features_dict.get('mouth_curve', 0.5) * 0.8 - 0.4  # [-0.4, 0.4]
    
    # Цвет лица (RGB) - от холодного (синий) к теплому (красный)
    face_temp = features_dict.get('face_temp', 0.5)
    if face_temp < 0.33:
        face_rgb = (0.7, 0.8, 1.0)  # светло-голубой
    elif face_temp < 0.66:
        face_rgb = (0.9, 0.9, 0.8)  # нейтральный/бежевый
    else:
        face_rgb = (1.0, 0.8, 0.7)  # розоватый/теплый
    
    # Рисуем овал лица
    face_oval = Circle((0, 0), 1.2, facecolor=face_rgb, edgecolor='black', 
                       linewidth=1, zorder=1)
    ax.add_patch(face_oval)
    
    # Рисуем левый глаз
    eye_L_x, eye_L_y = -0.6, 0.4
    eye_L = Circle((eye_L_x, eye_L_y), eye_size_L, facecolor='white', 
                   edgecolor='black', linewidth=1, zorder=2)
    ax.add_patch(eye_L)
    # Зрачок
    pupil_L = Circle((eye_L_x, eye_L_y), eye_size_L * 0.5, 
                     facecolor='black', zorder=3)
    ax.add_patch(pupil_L)
    
    # Рисуем правый глаз
    eye_R_x, eye_R_y = 0.6, 0.4
    eye_R = Circle((eye_R_x, eye_R_y), eye_size_R, facecolor='white', 
                   edgecolor='black', linewidth=1, zorder=2)
    ax.add_patch(eye_R)
    # Зрачок
    pupil_R = Circle((eye_R_x, eye_R_y), eye_size_R * 0.5, 
                     facecolor='black', zorder=3)
    ax.add_patch(pupil_R)
    
    # Рисуем левую бровь (линия с наклоном)
    brow_L_x = np.array([-0.9, -0.3]) + 0.1  # сдвиг для симметрии
    brow_L_y = np.array([0.8, 0.7]) + brow_slant_L * 0.4
    ax.plot(brow_L_x, brow_L_y, 'k-', linewidth=3, zorder=2)
    
    # Рисуем правую бровь
    brow_R_x = np.array([0.9, 0.3]) - 0.1  # сдвиг для симметрии
    brow_R_y = np.array([0.8, 0.7]) + brow_slant_R * 0.4
    ax.plot(brow_R_x, brow_R_y, 'k-', linewidth=3, zorder=2)
    
    # Рисуем нос (треугольник)
    nose_x = [0, -0.15, 0.15]
    nose_y = [0.1, -0.3 - nose_length, -0.3 - nose_length]
    nose = Polygon(np.column_stack([nose_x, nose_y]), facecolor='lightpink',
                   edgecolor='black', linewidth=1, zorder=2)
    ax.add_patch(nose)
    
    # Рисуем рот (изогнутая линия)
    mouth_x = np.linspace(-0.5, 0.5, 50)
    # Простая парабола для улыбки/грусти
    mouth_y = -0.6 + mouth_curve * (1 - (mouth_x/0.5)**2) * 0.3
    ax.plot(mouth_x, mouth_y, 'r-', linewidth=3, zorder=2)
    
    # Добавляем метку дня, если есть
    if day_label:
        ax.text(0, -1.6, day_label, ha='center', va='center', 
                fontsize=9, fontweight='bold')
    
    # Добавляем заголовок
    if title:
        ax.set_title(title, fontsize=10)

print("   Функция рисования лица создана.")


def create_features_mapping(row, feature_names):
    """
    Преобразует строку данных в словарь признаков для лица Чернова.
    
    Здесь мы определяем, какие параметры из набора данных
    будут отвечать за какие черты лица.
    
    Аргументы:
    - row: массив значений признаков для одного дня
    - feature_names: список названий признаков
    
    Возвращает:
    - словарь признаков для лица
    """
    features = {}
    
    # Создаем словарь для легкого доступа к значениям по имени
    value_dict = {}
    for i, name in enumerate(feature_names):
        value_dict[name] = row[i]
    
    # 1. Размер левого глаза: Thermostat Temp (температура в зоне)
    temp_cols = [name for name in feature_names if 'Thermostat Temp' in name]
    if temp_cols:
        features['eye_size_L'] = value_dict.get(temp_cols[0], 0.5)
    else:
        features['eye_size_L'] = 0.5
    
    # 2. Размер правого глаза: Equipment Power (активность оборудования)
    equip_cols = [name for name in feature_names if 'Equipment Power' in name]
    if equip_cols:
        features['eye_size_R'] = value_dict.get(equip_cols[0], 0.5)
    else:
        features['eye_size_R'] = 0.5
    
    # 3. Наклон левой брови: Lights Power (освещение)
    lights_cols = [name for name in feature_names if 'Lights Power' in name]
    if lights_cols:
        features['brow_slant_L'] = value_dict.get(lights_cols[0], 0.5)
    else:
        features['brow_slant_L'] = 0.5
    
    # 4. Наклон правой брови: Heating Setpoint (уставка отопления)
    heat_setpoint_cols = [name for name in feature_names if 'Heating Setpoint' in name]
    if heat_setpoint_cols:
        features['brow_slant_R'] = value_dict.get(heat_setpoint_cols[0], 0.5)
    else:
        features['brow_slant_R'] = 0.5
    
    # 5. Длина носа: Cooling Setpoint (уставка охлаждения)
    cool_setpoint_cols = [name for name in feature_names if 'Cooling Setpoint' in name]
    if cool_setpoint_cols:
        features['nose_length'] = value_dict.get(cool_setpoint_cols[0], 0.5)
    else:
        features['nose_length'] = 0.5
    
    # 6. Кривизна рта: комфортность (отклонение температуры от 0.5)
    if temp_cols:
        temp_val = value_dict.get(temp_cols[0], 0.5)
        # Чем ближе к 0.5, тем более улыбающееся лицо
        features['mouth_curve'] = 0.5 - abs(temp_val - 0.5)  # От 0 до 0.5
    else:
        features['mouth_curve'] = 0.3
    
    # 7. Цвет лица: Thermostat Temp (дублируем для цвета)
    if temp_cols:
        features['face_temp'] = value_dict.get(temp_cols[0], 0.5)
    else:
        features['face_temp'] = 0.5
    
    return features

print("   Функция маппинга признаков создана.")

## 5. Создание схемы визуализации (для отчета)

In [ ]:
print("\n" + "="*60)
print("5. СОЗДАНИЕ СХЕМЫ ВИЗУАЛИЗАЦИИ")
print("="*60)

# Создаем отдельную фигуру для схемы кодирования
fig_scheme, ax_scheme = plt.subplots(1, 1, figsize=(8, 8))

# Рисуем "эталонное" лицо со средними параметрами для демонстрации схемы
mean_features = {
    'eye_size_L': 0.6,
    'eye_size_R': 0.6,
    'brow_slant_L': 0.0,
    'brow_slant_R': 0.0,
    'nose_length': 0.4,
    'mouth_curve': 0.2,  # Немного улыбки
    'face_temp': 0.5
}

draw_chernoff_face(ax_scheme, mean_features)

# Находим названия параметров для аннотаций
temp_name = next((name.split(':')[-1].strip() for name in feature_cols if 'Thermostat Temp' in name), 'Температура')
equip_name = next((name.split(':')[-1].strip() for name in feature_cols if 'Equipment Power' in name), 'Активность')
lights_name = next((name.split(':')[-1].strip() for name in feature_cols if 'Lights Power' in name), 'Освещение')
heat_name = next((name.split(':')[-1].strip() for name in feature_cols if 'Heating Setpoint' in name), 'Уставка отопления')
cool_name = next((name.split(':')[-1].strip() for name in feature_cols if 'Cooling Setpoint' in name), 'Уставка охлаждения')

# Добавляем аннотации для каждой черты
ax_scheme.annotate(f'Размер левого глаза:\n{temp_name}', 
                   xy=(-0.6, 0.6), xytext=(-1.8, 1.0),
                   arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=9,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax_scheme.annotate(f'Размер правого глаза:\n{equip_name}', 
                   xy=(0.6, 0.6), xytext=(1.2, 1.0),
                   arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=9,
                   ha='left', bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax_scheme.annotate(f'Наклон левой брови:\n{lights_name}', 
                   xy=(-0.8, 1.0), xytext=(-1.8, 1.3),
                   arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=9,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax_scheme.annotate(f'Наклон правой брови:\n{heat_name}', 
                   xy=(0.8, 1.0), xytext=(1.2, 1.3),
                   arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=9,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax_scheme.annotate(f'Длина носа:\n{cool_name}', 
                   xy=(0.0, -0.1), xytext=(0.5, -1.0),
                   arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=9,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax_scheme.annotate(f'Кривизна рта:\nКомфортность\n(отклонение температуры)', 
                   xy=(0.0, -0.7), xytext=(-1.2, -1.5),
                   arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=9,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax_scheme.annotate(f'Цвет лица:\n{temp_name}', 
                   xy=(0.0, 0.2), xytext=(1.5, 0.0),
                   arrowprops=dict(arrowstyle='->', lw=1.5), fontsize=9,
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.suptitle('Схема кодирования параметров HVAC зоны F1_Z2 в лице Чернова', 
             fontsize=14, y=0.98)
plt.tight_layout()
plt.savefig('encoding_scheme.png', dpi=150, bbox_inches='tight')
plt.show()
print("   Схема кодирования сохранена в 'encoding_scheme.png'")

## 6. Построение всех 14 лиц (календарь)

In [ ]:
print("\n" + "="*60)
print("6. ПОСТРОЕНИЕ КАЛЕНДАРЯ ЛИЦ ЧЕРНОВА (14 ДНЕЙ)")
print("="*60)

# Проверяем, сколько дней у нас есть
n_days = len(normalized_df)
n_cols = 7
n_rows = (n_days + n_cols - 1) // n_cols  # Округление вверх

# Создаем фигуру с подграфиками
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.5))
axes = axes.flatten()  # Делаем плоским для удобства

# Для каждого дня строим лицо
for i, (idx, row) in enumerate(normalized_df.iterrows()):
    if i >= n_days:
        break
    
    # Получаем нормализованные значения (без даты)
    values = row[feature_cols].values
    
    # Создаем маппинг признаков
    features = create_features_mapping(values, feature_cols)
    
    # Формируем метку дня
    day_str = row['date'].strftime('%d %b')  # например, "31 May"
    
    # Рисуем лицо
    draw_chernoff_face(axes[i], features, day_label=day_str)

# Убираем лишние подграфики
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle(f'Визуализация параметров HVAC (зона F1_Z2) с помощью лиц Чернова', 
             fontsize=14, y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.savefig('chernoff_faces_calendar.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n   ✅ Визуализация построена и сохранена в 'chernoff_faces_calendar.png'")

## 7. Анализ результатов и ответы на вопросы

In [ ]:
print("\n" + "="*60)
print("7. АНАЛИЗ РЕЗУЛЬТАТОВ")
print("="*60)

# Создаем DataFrame с нормализованными значениями для анализа
analysis_df = normalized_df.copy()

# Создаем простой индекс комфорта на основе температуры
temp_col = next((col for col in feature_cols if 'Thermostat Temp' in col), None)
if temp_col:
    # Чем ближе температура к среднему (0.5), тем комфортнее
    analysis_df['comfort_index'] = 1 - np.abs(analysis_df[temp_col] - 0.5) * 2
else:
    analysis_df['comfort_index'] = 0.5

# Находим дни с аномалиями (низкий индекс комфорта)
threshold = analysis_df['comfort_index'].quantile(0.25)  # Нижний квартиль
anomaly_days = analysis_df[analysis_df['comfort_index'] <= threshold]

print(f"\n   Вопрос 1: В какие дни возникали аномальные ситуации?")
print(f"   Дни с потенциальными аномалиями (индекс комфорта <= {threshold:.3f}):")
if len(anomaly_days) > 0:
    for _, row in anomaly_days.iterrows():
        print(f"     - {row['date']} (комфорт: {row['comfort_index']:.3f})")
else:
    print("     - Аномалий не обнаружено по индексу комфорта.")

print(f"\n   Вопрос 2: Какие параметры связаны с аномалиями?")
print("   На основе анализа лиц Чернова и корреляций:")
print("   1. Температура (размер левого глаза) - резкое изменение указывает на дискомфорт")
print("   2. Активность оборудования (размер правого глаза) - указывает на загруженность помещения")
print("   3. Освещение (наклон левой брови) - может влиять на тепловыделение")
print("   4. Уставки температуры (длина носа и наклон правой брови) - настройки системы")

# Сохраняем результаты анализа
analysis_df.to_csv('daily_analysis_results.csv', index=False)
print("\n   ✅ Результаты анализа сохранены в 'daily_analysis_results.csv'")

print("\n" + "="*60)
print("РАБОТА ВЫПОЛНЕНА УСПЕШНО!")
print("="*60)
print("\n   Созданные файлы для отчета:")
print("   - correlation_matrix.png - матрица корреляции параметров")
print("   - encoding_scheme.png - схема кодирования лица Чернова (Рисунок 1)")
print("   - chernoff_faces_calendar.png - итоговая визуализация (Рисунок 2)")
print("   - daily_analysis_results.csv - результаты анализа по дням")
print("\n   Не забудьте указать свое ФИО в отчете!")